# Colab Setup

In [ ]:
import sys
import os
import torch
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, HTML

BASE_DIR = "/content/multilingual_bias"

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if os.path.isdir(BASE_DIR):
        import shutil
        print("Removing existing repo to update")
        shutil.rmtree(BASE_DIR)
  
    from git import Repo
    
    os.chdir("/")

    repo_url = "https://github.com/jonesmonez/multilingual_bias"
    Repo.clone_from(repo_url, BASE_DIR, branch="python312", single_branch=True)
    
    os.chdir(BASE_DIR)
    print(f"Repository cloned to {BASE_DIR}")

Repository cloned to /content/multilingual_bias


In [ ]:
load_dotenv()

value = os.getenv('KAGGLE_API_TOKEN')

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
api.dataset_download_files('dudewithpants/wiki-dump', path='.', unzip=True)   

import kaggle

# Crows Tests

In [2]:
from experiments.modules.crows_runner import CrowSPairsRunnerWrapper

runner = CrowSPairsRunnerWrapper()

to_test_types = ["gender", "race-color", "religion"]
# to_test_lang = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]
to_test_lang = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]
debias_lang = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [3]:
results_base = {}

for lang in to_test_lang:
    result_btype = {}
    
    for btype in to_test_types:
        
        result = runner.run_plain(
            path_to_crows=f"data/crows_improved/crows_{lang}.csv",
            lang_eval=lang,
            bias_type=btype,
        )
        
        result_btype[btype] = result[0]
    
    results_base[lang] = result_btype
    
print(results_base)
display(pd.DataFrame.from_dict(results_base, orient="index"))

  0%|          | 0/327 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

  0%|          | 0/88 [00:00<?, ?it/s]

  0%|          | 0/319 [00:00<?, ?it/s]

  0%|          | 0/502 [00:00<?, ?it/s]

...

  0%|          | 0/262 [00:00<?, ?it/s]

  0%|          | 0/510 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

{'ar_DZ': {'gender': 49.38, 'race-color': 62.42, 'religion': 57.65}, 'ca_ES': {'gender': 47.91, 'race-color': 50.34, 'religion': 64.29}, 'de_DE': {'gender': 52.3, 'race-color': 55.09, 'religion': 56.88}, 'en_US': {'gender': 51.9, 'race-color': 42.89, 'religion': 63.06}, 'es_AR': {'gender': 49.81, 'race-color': 60.84, 'religion': 67.68}, 'fr_FR': {'gender': 49.15, 'race-color': 50.11, 'religion': 67.54}, 'it_IT': {'gender': 38.02, 'race-color': 34.62, 'religion': 69.16}, 'mt_MT': {'gender': 53.48, 'race-color': 32.68, 'religion': 52.63}, 'zh_CN': {'gender': 38.76, 'race-color': 64.55, 'religion': 75.68}}


,gender,race-color,religion
ar_DZ,49.38,62.42,57.65
ca_ES,47.91,50.34,64.29
de_DE,52.30,55.09,56.88
en_US,51.90,42.89,63.06
es_AR,49.81,60.84,67.68
fr_FR,49.15,50.11,67.54
it_IT,38.02,34.62,69.16
mt_MT,53.48,32.68,52.63
zh_CN,38.76,64.55,75.68


In [ ]:
results_debias = {}

for dlang in debias_lang:
    result_lang = {}
    
    for lang in to_test_lang:
        result_btype = {}
        
        for btype in to_test_types:
            
            if btype == "race-color":
                btype_path = "racecolor"
            else:
                btype_path = btype
            
            result = runner.run_debias(
                path_to_crows=f"data/crows_improved/crows_{lang}.csv",
                lang_debias=dlang,
                lang_eval=lang,
                bias_type=btype,
                bias_direction=f"data/subspace/{btype_path}_subspace_{dlang}.pt",
            )
            
            result_btype[btype] = result[0]
            
        torch.cuda.empty_cache()
        
        result_lang[lang] = result_btype
    
    results_debias[dlang] = result_lang
    
    print(result_lang)
    display(pd.DataFrame.from_dict(result_lang, orient="index"))
    
print(results_debias)

# CDA & Dropout

In [2]:
from experiments.modules.debias_trainer import DropoutTrainer, CDATrainer

ModuleNotFoundError: No module named 'experiments'

In [3]:
dropout_trainer = DropoutTrainer(
    model_name_or_path="bert-base-multilingual-uncased",
    max_seq_length=128,
    seed=42,
    fp16=True,
)

In [14]:
dropout_model_dir = dropout_trainer.train(
    train_file="wiki_de_DE.txt",
    output_dir="./debias_models/bert-base-uncased_dropout",
    num_train_epochs=3,
    per_device_train_batch_size=24,
    learning_rate=3e-5,
    logging_steps=200,
    save_steps=500,
    warmup_steps=500,
    max_grad_norm=1.0,
)

In [5]:
print(dropout_model_dir)

debias_models/bert-base-uncased_dropout


# INLP

In [1]:
from experiments.modules.inlp_runner import InlpRunner
runner = InlpRunner(
    "BertModel",
    "bert-base-multilingual-uncased",
)
import nltk
nltk.download('punkt_tab')
runner.setup_data(
    "wiki_de_DE.txt",
    "data/bias_attribute/translated/bias_attribute_words_de_DE.json",
    "de_DE",
    "religion"
)

ModuleNotFoundError: No module named 'experiments'

In [10]:
runner.compute_projection_matrix()

Encoding neutral sentences: 100%|██████████| 10000/10000 [02:10<00:00, 76.40it/s]


Dataset split sizes:
Train size: 9800; Dev size: 4200; Test size: 6000


iteration: 79, accuracy: 0.5573809523809524: 100%|██████████| 80/80 [07:29<00:00,  5.62s/it]


tensor([[ 8.8124e-01, -2.2041e-04, -8.0665e-03,  ..., -3.9003e-03,
          4.7755e-04,  2.6445e-03],
        [-2.2041e-04,  8.9042e-01,  2.7494e-02,  ..., -1.1926e-02,
          9.7388e-04,  9.2103e-05],
        [-8.0665e-03,  2.7494e-02,  8.3932e-01,  ..., -4.9165e-02,
         -4.1225e-03, -7.6228e-03],
        ...,
        [-3.9003e-03, -1.1926e-02, -4.9165e-02,  ...,  8.8629e-01,
         -7.1935e-03, -5.8769e-03],
        [ 4.7755e-04,  9.7388e-04, -4.1225e-03,  ..., -7.1935e-03,
          9.2841e-01,  4.9368e-03],
        [ 2.6445e-03,  9.2103e-05, -7.6228e-03,  ..., -5.8769e-03,
          4.9368e-03,  9.2423e-01]])